#Transformed Circuits data
- Read bronze circuits table
- Keep only the columns required for analytics (Drop url column)
- Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
- Rename columns to make them more meaningful (lat → latitude, lng → longitude)
- Filter out rows where circuit_id is null (business key validation)
- Remove duplicate records
- Transform values of columns circuit_name and locality to Title Case
- Write the transformed data to silver circuits table

In [0]:
%run ../00-common/01.environment.config


In [0]:
%python
bronze_table = f"{catalog_name}.{bronze_schema_name}.circuits"
silver_table = f"{catalog_name}.{silver_schema_name}.circuits"

#Read bronze circuits table

In [0]:
%python
circuits_df = spark.read.table(bronze_table)

In [0]:
%python
display(circuits_df)
       


circuitId,url,circuitName,lat,long,locality,country,ingestion_timestamp,source_file
null,https://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve,circuit gilles villeneuve,45.5,-73.5228,montreal,Canada,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
null,https://en.wikipedia.org/wiki/Lusail_International_Circuit,losail international circuit,25.49,51.4542,lusail,Qatar,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
adelaide,https://en.wikipedia.org/wiki/Adelaide_Street_Circuit,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
ain-diab,https://en.wikipedia.org/wiki/Ain-Diab_Circuit,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
aintree,https://en.wikipedia.org/wiki/Aintree_Motor_Racing_Circuit,aintree,53.4769,-2.94056,liverpool,UK,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
albert_park,https://en.wikipedia.org/wiki/Albert_Park_Circuit,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
americas,https://en.wikipedia.org/wiki/Circuit_of_the_Americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,https://en.wikipedia.org/wiki/Anderstorp_Raceway,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
avus,https://en.wikipedia.org/wiki/AVUS,avus,52.4806,13.2514,berlin,Germany,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
bahrain,https://en.wikipedia.org/wiki/Bahrain_International_Circuit,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


#Keep only the columns required for analytics (Drop url column)

In [0]:
%python
 circuits_selected_df = circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file"
)

#Standardise column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
#Rename columns to make them more meaningful (lat → latitude, lng → longitude)

In [0]:
%python
circuits_renamed_df = (circuits_selected_df.withColumnRenamed("circuitId", "circuit_id")
.withColumnRenamed("lat", "latitude")
.withColumnRenamed("long", "longitude")
.withColumnRenamed("circuitName", "circuit_name")
)

#Filter out rows where circuit_id is null (business key validation)

In [0]:
%python
circuits_valid_df = circuits_renamed_df.filter(
"circuit_id IS NOT NULL"
)

#Remove duplicate records

In [0]:
%python
circuits_distinct_df = circuits_valid_df.distinct()

#Transform values of columns circuit_name and locality to Title Case

In [0]:
%python
from pyspark.sql import functions as F
circuits_final_df = (circuits_distinct_df
                     .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
                     .withColumn("locality", F.initcap(F.col("locality")))

)

In [0]:
%python
display(circuits_final_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
madring,Madring,null,null,Madrid,Spain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
galvez,Autódromo Juan Y Oscar Gálvez,-34.6943,-58.4593,Buenos Aires,Argentina,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
interlagos,Autódromo José Carlos Pace,-23.7036,-46.6997,São Paulo,Brazil,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
miami,Miami International Autodrome,25.9581,-80.2389,Miami,USA,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
nurburgring,Nürburgring,50.3356,6.9475,Nürburg,Germany,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
tremblant,Circuit Mont-tremblant,46.1877,-74.6099,Quebec,Canada,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
catalunya,Circuit De Barcelona-catalunya,41.57,2.26111,Barcelona,Spain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
donington,Donington Park,52.8306,-1.37528,Castle Donington,UK,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
kyalami,Kyalami,-25.9894,28.0767,Midrand,South Africa,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv


#Write the transformed data to silver circuits table

In [0]:
%python
(
    circuits_final_df
    .write
    .format("delta") 
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
%python
display(spark.table(silver_table))

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file
madring,Madring,null,null,Madrid,Spain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
galvez,Autódromo Juan Y Oscar Gálvez,-34.6943,-58.4593,Buenos Aires,Argentina,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
interlagos,Autódromo José Carlos Pace,-23.7036,-46.6997,São Paulo,Brazil,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
miami,Miami International Autodrome,25.9581,-80.2389,Miami,USA,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
nurburgring,Nürburgring,50.3356,6.9475,Nürburg,Germany,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
tremblant,Circuit Mont-tremblant,46.1877,-74.6099,Quebec,Canada,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
catalunya,Circuit De Barcelona-catalunya,41.57,2.26111,Barcelona,Spain,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
donington,Donington Park,52.8306,-1.37528,Castle Donington,UK,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
kyalami,Kyalami,-25.9894,28.0767,Midrand,South Africa,2026-06-15T21:46:06.402994Z,dbfs:/Volumes/formula1/landing/files/circuits.csv
